# Laboratorio 07 - Soluciones: Storytelling con Datos

**Soluciones completas y ejemplos de narrativas bien construidas**

---

## Objetivo de este documento

Este notebook proporciona:
1. Código completo para todos los ejercicios
2. Ejemplos de narrativas bien estructuradas
3. Visualizaciones finales con todos los elementos de storytelling
4. Rúbrica de evaluación detallada

---

## Parte 0: Configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Configuración para gráficos limpios
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['font.family'] = 'sans-serif'

# Colores consistentes (paleta profesional)
COLOR_PRINCIPAL = '#2563EB'  # Azul
COLOR_ACENTO = '#DC2626'     # Rojo
COLOR_SECUNDARIO = '#94A3B8' # Gris
COLOR_POSITIVO = '#059669'   # Verde
COLOR_NEGATIVO = '#DC2626'   # Rojo
COLOR_FONDO = '#F8FAFC'      # Gris muy claro

print("Configuración lista")

In [ ]:
# Cargar datos
try:
    url_matricula = "https://raw.githubusercontent.com/gonzalezulises/formacion-docente-bi-faces/main/datasets/universidad/matricula_faces.csv"
    df_matricula = pd.read_csv(url_matricula)
    print(f"Matrícula cargada: {df_matricula.shape[0]} registros")
except:
    # Crear datos de ejemplo si no está disponible
    np.random.seed(42)
    años = list(range(2019, 2025))
    carreras = ['Administración', 'Contaduría', 'Economía', 'RRII', 'Sociología']
    
    data = []
    base_matricula = {'Administración': 1200, 'Contaduría': 950, 'Economía': 800, 
                      'RRII': 700, 'Sociología': 400}
    
    for año in años:
        for carrera in carreras:
            # Tendencia decreciente con variación por carrera
            factor = 1 - 0.03 * (año - 2019) + np.random.uniform(-0.02, 0.02)
            if carrera == 'Economía':
                factor -= 0.02 * (año - 2019)  # Mayor caída en Economía
            elif carrera == 'Administración':
                factor += 0.01 * (año - 2019)  # Menor caída en Admin
            
            cantidad = int(base_matricula[carrera] * factor)
            data.append({'periodo': f'{año}-1', 'carrera': carrera, 'cantidad': cantidad})
            data.append({'periodo': f'{año}-2', 'carrera': carrera, 
                        'cantidad': int(cantidad * 0.95)})
    
    df_matricula = pd.DataFrame(data)
    print(f"Datos de ejemplo creados: {df_matricula.shape[0]} registros")

# Preparar datos
df_matricula['año'] = df_matricula['periodo'].str[:4].astype(int)
df_matricula['semestre'] = df_matricula['periodo'].str[-1].astype(int)

df_matricula.head()

---

## Solución Ejercicio 1.1: Identificar la Historia

In [ ]:
# SOLUCIÓN: Análisis de tendencia de matrícula

# Agrupar por año
tendencia = df_matricula.groupby('año')['cantidad'].sum().reset_index()

# Calcular cambio porcentual
tendencia['cambio_pct'] = tendencia['cantidad'].pct_change() * 100

# Calcular cambio total
primer_año = tendencia['cantidad'].iloc[0]
ultimo_año = tendencia['cantidad'].iloc[-1]
cambio_total = (ultimo_año - primer_año) / primer_año * 100

# Encontrar año con mayor caída
año_peor = tendencia.loc[tendencia['cambio_pct'].idxmin()]

print("=" * 60)
print("ANÁLISIS DE TENDENCIA DE MATRÍCULA")
print("=" * 60)
print(f"\nPeríodo analizado: {tendencia['año'].min()} - {tendencia['año'].max()}")
print(f"\nMatrícula inicial ({tendencia['año'].iloc[0]}): {primer_año:,}")
print(f"Matrícula final ({tendencia['año'].iloc[-1]}): {ultimo_año:,}")
print(f"\nCambio total: {cambio_total:+.1f}%")
print(f"Pérdida de estudiantes: {primer_año - ultimo_año:,}")
print(f"\nAño con mayor caída: {int(año_peor['año'])} ({año_peor['cambio_pct']:.1f}%)")
print("\n" + "=" * 60)
print(tendencia.to_string(index=False))

### Respuestas a Pregunta de Reflexión 1.1

1. **Tendencia principal**: Decreciente - la matrícula ha caído aproximadamente 15-20% en el período analizado.

2. **Punto de inflexión**: El año 2020 muestra la mayor caída interanual, probablemente relacionado con la pandemia de COVID-19.

3. **Historia posible**: 
   > "La matrícula de FACES ha perdido más de 600 estudiantes en 5 años, una caída del 15%. Si esta tendencia continúa, en 3 años habremos perdido el equivalente a una escuela completa. Sin embargo, algunas carreras como Administración muestran signos de recuperación, lo que sugiere que intervenciones focalizadas pueden revertir la tendencia."

---

## Solución Ejercicio 2.1 y 2.2: Versión Típica vs Storytelling

In [ ]:
# SOLUCIÓN: Comparación de versiones

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ====== VERSIÓN TÍPICA (MALA) ======
ax1 = axes[0]
ax1.plot(tendencia['año'], tendencia['cantidad'], marker='o')
ax1.set_title('Matrícula por Año')
ax1.set_xlabel('Año')
ax1.set_ylabel('Cantidad')
ax1.grid(True)

# Añadir texto de "MAL"
ax1.text(0.5, 0.95, 'VERSIÓN TÍPICA', transform=ax1.transAxes,
         ha='center', va='top', fontsize=14, color='red', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#ffebee', edgecolor='red'))

# ====== VERSIÓN STORYTELLING (BUENA) ======
ax2 = axes[1]

# Color según tendencia
color_linea = COLOR_NEGATIVO if cambio_total < 0 else COLOR_POSITIVO

# Línea principal
ax2.plot(tendencia['año'], tendencia['cantidad'], 
         marker='o', linewidth=2.5, markersize=10,
         color=color_linea, markerfacecolor='white', 
         markeredgewidth=2, markeredgecolor=color_linea)

# Área sombreada para mostrar pérdida
ax2.fill_between(tendencia['año'], tendencia['cantidad'], primer_año,
                 alpha=0.15, color=COLOR_NEGATIVO)

# Línea de referencia (nivel inicial)
ax2.axhline(y=primer_año, color=COLOR_SECUNDARIO, linestyle='--', 
            alpha=0.7, linewidth=1.5)
ax2.text(tendencia['año'].max() + 0.1, primer_año, 
         f'Meta: {primer_año:,}', va='center', fontsize=10, color=COLOR_SECUNDARIO)

# Anotación en punto crítico
idx_peor = tendencia['cambio_pct'].idxmin()
ax2.annotate(f'Mayor caída\n{año_peor["cambio_pct"]:.1f}%',
             xy=(año_peor['año'], año_peor['cantidad']),
             xytext=(año_peor['año'] - 0.8, año_peor['cantidad'] - 150),
             fontsize=10, ha='center',
             arrowprops=dict(arrowstyle='->', color=COLOR_ACENTO, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLOR_ACENTO))

# Título con MENSAJE
ax2.set_title(f'La matrícula de FACES ha caído {abs(cambio_total):.0f}% desde {tendencia["año"].iloc[0]}',
              fontsize=14, fontweight='bold', color='#1e293b', pad=20)

# Subtítulo con contexto
estudiantes_perdidos = primer_año - ultimo_año
ax2.text(0.5, 1.02, f'Pérdida de {estudiantes_perdidos:,} estudiantes en {len(tendencia)-1} años',
         transform=ax2.transAxes, ha='center', fontsize=11, color='gray')

# Etiquetas de datos en puntos clave
for i in [0, -1]:  # Primer y último punto
    ax2.annotate(f'{tendencia["cantidad"].iloc[i]:,}',
                 xy=(tendencia['año'].iloc[i], tendencia['cantidad'].iloc[i]),
                 xytext=(0, 15), textcoords='offset points',
                 ha='center', fontsize=11, fontweight='bold')

# Limpiar
ax2.set_xlabel('')
ax2.set_ylabel('Estudiantes matriculados', fontsize=11, color='#64748b')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_ylim(tendencia['cantidad'].min() * 0.9, primer_año * 1.05)

# Fuente de datos
ax2.text(0.99, -0.12, 'Fuente: Sistema de Control de Estudios, FACES-UCAB',
         transform=ax2.transAxes, ha='right', fontsize=9, color='gray')

# Añadir texto de "BIEN"
ax2.text(0.5, 0.95, 'VERSIÓN STORYTELLING', transform=ax2.transAxes,
         ha='center', va='top', fontsize=14, color='green', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#e8f5e9', edgecolor='green'))

plt.tight_layout()
plt.show()

---

## Solución Ejercicio 3.1: Hook para la Apertura

In [ ]:
# SOLUCIÓN: Encontrar datos para hook

# Opción 1: Comparación impactante
estudiantes_perdidos = primer_año - ultimo_año
secciones_equivalentes = estudiantes_perdidos // 35  # Asumiendo 35 por sección

print("OPCIONES DE HOOK:")
print("="*60)
print(f"\n1. Comparación: 'Hemos perdido el equivalente a {secciones_equivalentes} secciones completas'")

# Opción 2: Proyección futura
# Calcular tasa de caída promedio
tasa_caida_anual = cambio_total / (len(tendencia) - 1)
años_hasta_50pct = int(50 / abs(tasa_caida_anual))
print(f"\n2. Proyección: 'Al ritmo actual, en {años_hasta_50pct} años tendremos la mitad de estudiantes'")

# Opción 3: La carrera más afectada
por_carrera = df_matricula.groupby(['carrera', 'año'])['cantidad'].sum().unstack()
cambio_carrera = (por_carrera.iloc[:, -1] - por_carrera.iloc[:, 0]) / por_carrera.iloc[:, 0] * 100
carrera_peor = cambio_carrera.idxmin()
caida_peor = cambio_carrera.min()
print(f"\n3. Dato específico: '{carrera_peor} ha perdido {abs(caida_peor):.0f}% de su matrícula'")

# Opción 4: Pregunta provocativa
print(f"\n4. Pregunta: '¿Sabían que cada año perdemos {int(estudiantes_perdidos / (len(tendencia)-1))} estudiantes?'")

### Hook seleccionado

**Mi hook:**

> "En los últimos 5 años, FACES ha perdido el equivalente a 17 secciones completas. Eso son 600 estudiantes que ya no están en nuestras aulas. Si no actuamos, en 10 años seremos la mitad."

---

## Solución Ejercicio 3.2: Los 3 Hallazgos

In [ ]:
# SOLUCIÓN: Desarrollar los 3 hallazgos

print("HALLAZGO 1: TENDENCIA GENERAL")
print("="*60)
print(f"Afirmación: La matrícula total ha caído {abs(cambio_total):.0f}% en {len(tendencia)-1} años")
print(f"Evidencia: De {primer_año:,} estudiantes en {tendencia['año'].iloc[0]} a {ultimo_año:,} en {tendencia['año'].iloc[-1]}")
print(f"Implicación: Pérdida de ingresos por matrícula y riesgo de cierre de secciones")

print("\n" + "="*60)
print("HALLAZGO 2: DIFERENCIAS POR CARRERA")
print("="*60)

# Análisis por carrera
cambio_carrera_sorted = cambio_carrera.sort_values()
print("\nCambio porcentual por carrera:")
for carrera, cambio in cambio_carrera_sorted.items():
    simbolo = "" if cambio >= 0 else ""
    print(f"  {simbolo} {carrera}: {cambio:+.1f}%")

carrera_mejor = cambio_carrera.idxmax()
cambio_mejor = cambio_carrera.max()

print(f"\nAfirmación: {carrera_peor} es la carrera más afectada, mientras {carrera_mejor} muestra signos de recuperación")
print(f"Evidencia: {carrera_peor}: {caida_peor:.1f}% vs {carrera_mejor}: {cambio_mejor:+.1f}%")
print(f"Implicación: Las intervenciones deben ser diferenciadas por carrera")

print("\n" + "="*60)
print("HALLAZGO 3: PATRÓN TEMPORAL")
print("="*60)

# Análisis por semestre
por_semestre = df_matricula.groupby(['año', 'semestre'])['cantidad'].sum().unstack()
caida_sem1 = (por_semestre[1].iloc[-1] - por_semestre[1].iloc[0]) / por_semestre[1].iloc[0] * 100
caida_sem2 = (por_semestre[2].iloc[-1] - por_semestre[2].iloc[0]) / por_semestre[2].iloc[0] * 100

print(f"\nCaída en Semestre 1: {caida_sem1:.1f}%")
print(f"Caída en Semestre 2: {caida_sem2:.1f}%")

print(f"\nAfirmación: La caída es más pronunciada en el segundo semestre")
print(f"Evidencia: Sem1: {caida_sem1:.1f}% vs Sem2: {caida_sem2:.1f}%")
print(f"Implicación: Los programas de retención deben focalizarse en el primer semestre")

---

## Solución Ejercicio 4.1: Visualización para Hallazgo 1 (Tendencia)

In [ ]:
# SOLUCIÓN: Visualización de tendencia con storytelling completo

fig, ax = plt.subplots(figsize=(14, 8))

# Datos
años = tendencia['año'].values
valores = tendencia['cantidad'].values

# Proyección futura (3 años)
años_futuros = np.array([años[-1] + 1, años[-1] + 2, años[-1] + 3])
slope, intercept, _, _, _ = stats.linregress(años, valores)
proyeccion = intercept + slope * años_futuros

# Área de pérdida (sombreado)
ax.fill_between(años, valores, primer_año, 
                alpha=0.15, color=COLOR_NEGATIVO, label='Estudiantes perdidos')

# Línea histórica
ax.plot(años, valores, 
        marker='o', linewidth=3, markersize=12,
        color=COLOR_NEGATIVO, markerfacecolor='white', 
        markeredgewidth=2.5, markeredgecolor=COLOR_NEGATIVO,
        label='Matrícula real', zorder=10)

# Proyección futura
ax.plot(años_futuros, proyeccion, 
        linestyle='--', linewidth=2, color=COLOR_ACENTO, alpha=0.7,
        marker='o', markersize=8, markerfacecolor='white',
        label='Proyección si continúa tendencia')

# Línea de meta
ax.axhline(y=primer_año, color=COLOR_SECUNDARIO, linestyle=':', 
           linewidth=2, alpha=0.8)
ax.text(años[-1] + 3.2, primer_año, f'Meta:\n{primer_año:,}', 
        va='center', fontsize=11, color=COLOR_SECUNDARIO, fontweight='bold')

# Anotaciones en puntos clave
# Punto inicial
ax.annotate(f'{valores[0]:,}\nestudiantes',
            xy=(años[0], valores[0]),
            xytext=(años[0] - 0.5, valores[0] + 80),
            fontsize=11, ha='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLOR_SECUNDARIO))

# Punto final
ax.annotate(f'{valores[-1]:,}\nestudiantes',
            xy=(años[-1], valores[-1]),
            xytext=(años[-1] + 0.5, valores[-1] - 100),
            fontsize=11, ha='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLOR_NEGATIVO))

# Proyección final
ax.annotate(f'Proyección {años_futuros[-1]}:\n{int(proyeccion[-1]):,}',
            xy=(años_futuros[-1], proyeccion[-1]),
            xytext=(años_futuros[-1], proyeccion[-1] - 120),
            fontsize=10, ha='center', color=COLOR_ACENTO,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffebee', edgecolor=COLOR_ACENTO, alpha=0.9))

# Flecha mostrando la caída
ax.annotate('', xy=(años[-1] + 0.15, valores[-1]), 
            xytext=(años[-1] + 0.15, primer_año),
            arrowprops=dict(arrowstyle='<->', color=COLOR_NEGATIVO, lw=2))
ax.text(años[-1] + 0.35, (primer_año + valores[-1])/2, 
        f'-{abs(cambio_total):.0f}%\n({primer_año - valores[-1]:,})',
        fontsize=11, va='center', color=COLOR_NEGATIVO, fontweight='bold')

# Título con mensaje
ax.set_title(f'La matrícula de FACES ha caído {abs(cambio_total):.0f}% desde {años[0]}',
             fontsize=18, fontweight='bold', color='#1e293b', pad=25)

# Subtítulo
ax.text(0.5, 1.03, f'Sin intervención, perderemos {int(valores[-1] - proyeccion[-1]):,} estudiantes adicionales para {años_futuros[-1]}',
        transform=ax.transAxes, ha='center', fontsize=12, color='#64748b')

# Configuración de ejes
ax.set_xlabel('')
ax.set_ylabel('Estudiantes matriculados', fontsize=12, color='#64748b')
ax.set_xlim(años[0] - 0.5, años_futuros[-1] + 0.8)
ax.set_ylim(min(proyeccion) * 0.92, primer_año * 1.08)

# Limpiar
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Leyenda
ax.legend(loc='lower left', fontsize=10, framealpha=0.9)

# Fuente
ax.text(0.99, -0.08, 'Fuente: Sistema de Control de Estudios, FACES-UCAB | Análisis: Enero 2025',
        transform=ax.transAxes, ha='right', fontsize=9, color='#94a3b8')

plt.tight_layout()
plt.savefig('hallazgo_1_tendencia.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n Visualización guardada como 'hallazgo_1_tendencia.png'")

---

## Solución Ejercicio 4.2: Visualización para Hallazgo 2 (Comparación por Carrera)

In [ ]:
# SOLUCIÓN: Comparación entre carreras

fig, ax = plt.subplots(figsize=(12, 7))

# Datos ordenados
datos = cambio_carrera_sorted

# Colores según signo
colores = [COLOR_POSITIVO if x >= 0 else COLOR_NEGATIVO for x in datos]

# Barras horizontales
bars = ax.barh(range(len(datos)), datos.values, color=colores, height=0.6, alpha=0.85)

# Línea de referencia en 0
ax.axvline(x=0, color='#1e293b', linewidth=1.5)

# Etiquetas de carreras
ax.set_yticks(range(len(datos)))
ax.set_yticklabels(datos.index, fontsize=12)

# Etiquetas de valores
for i, (carrera, valor) in enumerate(datos.items()):
    # Posición del texto
    offset = 1.5 if valor >= 0 else -1.5
    ha = 'left' if valor >= 0 else 'right'
    
    # Texto con formato
    texto = f'{valor:+.1f}%'
    ax.text(valor + offset, i, texto, va='center', ha=ha,
            fontweight='bold', fontsize=11,
            color=COLOR_POSITIVO if valor >= 0 else COLOR_NEGATIVO)

# Resaltar la peor carrera
idx_peor = list(datos.index).index(carrera_peor)
bars[idx_peor].set_edgecolor('#1e293b')
bars[idx_peor].set_linewidth(2)

# Anotación explicativa
ax.annotate(f'{carrera_peor} requiere\natención urgente',
            xy=(datos[carrera_peor] - 2, idx_peor),
            xytext=(datos[carrera_peor] - 15, idx_peor + 0.5),
            fontsize=10, color=COLOR_NEGATIVO,
            arrowprops=dict(arrowstyle='->', color=COLOR_NEGATIVO, lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLOR_NEGATIVO))

# Título con mensaje
ax.set_title(f'{carrera_peor} ha perdido {abs(caida_peor):.0f}% de su matrícula,\nmientras {carrera_mejor} muestra recuperación',
             fontsize=14, fontweight='bold', color='#1e293b', pad=20)

# Subtítulo
ax.text(0.5, 1.02, f'Cambio porcentual de matrícula {años[0]}-{años[-1]}',
        transform=ax.transAxes, ha='center', fontsize=11, color='#64748b')

# Configuración
ax.set_xlabel('Cambio porcentual (%)', fontsize=11, color='#64748b')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# Expandir límites para etiquetas
ax.set_xlim(min(datos) * 1.3, max(datos) * 1.5 if max(datos) > 0 else 5)

# Fondo para área negativa
ax.axvspan(min(datos) * 1.3, 0, alpha=0.05, color=COLOR_NEGATIVO)

# Fuente
ax.text(0.99, -0.08, 'Fuente: Sistema de Control de Estudios, FACES-UCAB',
        transform=ax.transAxes, ha='right', fontsize=9, color='#94a3b8')

plt.tight_layout()
plt.savefig('hallazgo_2_carreras.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n Visualización guardada como 'hallazgo_2_carreras.png'")

---

## Solución Ejercicio 4.3: Visualización para Hallazgo 3 (Patrón Temporal)

In [ ]:
# SOLUCIÓN: Patrón por semestre

fig, ax = plt.subplots(figsize=(12, 7))

# Datos por año y semestre
ancho = 0.35
x = np.arange(len(por_semestre))

# Barras agrupadas
bars1 = ax.bar(x - ancho/2, por_semestre[1], ancho, 
               label='Semestre 1', color=COLOR_PRINCIPAL, alpha=0.85)
bars2 = ax.bar(x + ancho/2, por_semestre[2], ancho, 
               label='Semestre 2', color=COLOR_ACENTO, alpha=0.85)

# Líneas de tendencia
z1 = np.polyfit(x, por_semestre[1], 1)
p1 = np.poly1d(z1)
ax.plot(x, p1(x), '--', color=COLOR_PRINCIPAL, linewidth=2, alpha=0.7)

z2 = np.polyfit(x, por_semestre[2], 1)
p2 = np.poly1d(z2)
ax.plot(x, p2(x), '--', color=COLOR_ACENTO, linewidth=2, alpha=0.7)

# Anotaciones de pendiente
ax.text(x[-1] + 0.3, p1(x[-1]), f'{z1[0]:.0f}/año', 
        color=COLOR_PRINCIPAL, fontsize=10, fontweight='bold', va='center')
ax.text(x[-1] + 0.3, p2(x[-1]), f'{z2[0]:.0f}/año', 
        color=COLOR_ACENTO, fontsize=10, fontweight='bold', va='center')

# Configuración de ejes
ax.set_xticks(x)
ax.set_xticklabels(por_semestre.index)
ax.set_xlabel('Año', fontsize=11, color='#64748b')
ax.set_ylabel('Estudiantes matriculados', fontsize=11, color='#64748b')

# Título con mensaje
ax.set_title('La caída es más pronunciada en el segundo semestre',
             fontsize=14, fontweight='bold', color='#1e293b', pad=20)

# Subtítulo
ax.text(0.5, 1.02, f'Semestre 1: {caida_sem1:.1f}% vs Semestre 2: {caida_sem2:.1f}%',
        transform=ax.transAxes, ha='center', fontsize=11, color='#64748b')

# Limpiar
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, por_semestre.max().max() * 1.15)

# Leyenda
ax.legend(loc='upper right', fontsize=11)

# Insight box
textstr = 'Implicación: Los programas de\nretención deben focalizarse\nen el primer semestre'
props = dict(boxstyle='round,pad=0.5', facecolor='#f0f9ff', edgecolor=COLOR_PRINCIPAL, alpha=0.9)
ax.text(0.02, 0.02, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='bottom', bbox=props, color='#1e293b')

# Fuente
ax.text(0.99, -0.08, 'Fuente: Sistema de Control de Estudios, FACES-UCAB',
        transform=ax.transAxes, ha='right', fontsize=9, color='#94a3b8')

plt.tight_layout()
plt.savefig('hallazgo_3_semestres.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n Visualización guardada como 'hallazgo_3_semestres.png'")

---

## Solución Ejercicio 5.1: Recomendaciones

### Recomendación 1: Programa de Tutorías para Primer Año

| Campo | Contenido |
|-------|----------|
| **Acción** | Implementar programa de tutorías obligatorias para estudiantes de primer semestre |
| **Justificación** | Hallazgo 3 muestra que la mayor caída ocurre en el segundo semestre, lo que indica deserción temprana |
| **Indicador de éxito** | Reducir la tasa de deserción de primer a segundo semestre en 20% |
| **Horizonte temporal** | Piloto en Semestre 2025-1, evaluación en 2025-2 |

### Recomendación 2: Intervención Focalizada en Economía

| Campo | Contenido |
|-------|----------|
| **Acción** | Realizar estudio cualitativo en la Escuela de Economía para identificar causas específicas y rediseñar malla curricular |
| **Justificación** | Hallazgo 2 muestra que Economía tiene la mayor caída (-30%), muy por encima del promedio |
| **Indicador de éxito** | Revertir la tendencia negativa a neutral (0%) en 2 años |
| **Horizonte temporal** | Estudio en Semestre 2025-1, implementación de cambios en 2025-2 |

### Recomendación 3: Monitoreo Temprano de Riesgo

| Campo | Contenido |
|-------|----------|
| **Acción** | Desarrollar dashboard de indicadores tempranos de deserción (asistencia, notas parciales, uso de plataforma) |
| **Justificación** | Actualmente no existe sistema de alerta que permita intervención preventiva |
| **Indicador de éxito** | Identificar 80% de desertores potenciales antes del fin del primer parcial |
| **Horizonte temporal** | Desarrollo en 3 meses, piloto en Semestre 2025-1 |

---

## Solución Ejercicio 6.1: Diseño de los 5 Slides

### Slide 1 - Apertura
- **Hook**: "600 estudiantes menos. 17 secciones vacías. ¿Qué está pasando en FACES?"
- **Visual**: Número grande "600" en rojo con icono de persona tachado

### Slide 2 - Contexto
- **Texto**: "Analizamos 5 años de datos de matrícula para entender la tendencia"
- **Visual**: Gráfico de línea simple mostrando la caída

### Slide 3 - Hallazgo Principal
- **Mensaje**: "La matrícula ha caído 15% y seguirá cayendo si no actuamos"
- **Visual**: Gráfico de tendencia con proyección (hallazgo_1_tendencia.png)

### Slide 4 - Hallazgos Secundarios
- **Mensajes**: "Economía es la más afectada" + "El segundo semestre pierde más"
- **Visual**: Dos gráficos lado a lado (carreras + semestres)

### Slide 5 - Llamado a la Acción
- **Recomendaciones**: 3 acciones priorizadas con responsables
- **Próximo paso**: "Solicitamos aprobación para piloto de tutorías en Semestre 2025-1"

In [ ]:
# SOLUCIÓN: Slide de cierre (llamado a la acción)

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

# Título
ax.text(0.5, 0.95, 'Recomendaciones', 
        ha='center', va='top', fontsize=24, fontweight='bold', color='#1e293b')

# Subtítulo
ax.text(0.5, 0.88, 'Tres acciones para revertir la tendencia de matrícula',
        ha='center', va='top', fontsize=14, color='#64748b')

# Recomendaciones
recomendaciones = [
    ('1', 'Programa de Tutorías', 'Tutorías obligatorias para primer semestre', 
     'Reducir deserción temprana en 20%', COLOR_PRINCIPAL),
    ('2', 'Intervención en Economía', 'Estudio cualitativo y rediseño curricular',
     'Revertir caída de -30% a 0%', COLOR_ACENTO),
    ('3', 'Dashboard de Alerta', 'Sistema de indicadores tempranos de deserción',
     'Identificar 80% de riesgo antes de parciales', COLOR_POSITIVO)
]

for i, (num, titulo, descripcion, meta, color) in enumerate(recomendaciones):
    y_pos = 0.72 - i * 0.22
    
    # Círculo con número
    circle = plt.Circle((0.08, y_pos), 0.04, color=color, transform=ax.transAxes, 
                         clip_on=False)
    ax.add_patch(circle)
    ax.text(0.08, y_pos, num, ha='center', va='center', 
            fontsize=18, fontweight='bold', color='white')
    
    # Título de recomendación
    ax.text(0.15, y_pos + 0.03, titulo, ha='left', va='bottom',
            fontsize=16, fontweight='bold', color='#1e293b')
    
    # Descripción
    ax.text(0.15, y_pos - 0.02, descripcion, ha='left', va='top',
            fontsize=12, color='#64748b')
    
    # Meta
    ax.text(0.85, y_pos, f'Meta: {meta}', ha='right', va='center',
            fontsize=11, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                     edgecolor=color, alpha=0.9))

# Caja de próximo paso
rect = mpatches.FancyBboxPatch((0.1, 0.05), 0.8, 0.12,
                                boxstyle="round,pad=0.02,rounding_size=0.02",
                                facecolor='#f0fdf4', edgecolor=COLOR_POSITIVO,
                                linewidth=2, transform=ax.transAxes)
ax.add_patch(rect)

ax.text(0.5, 0.11, 'Próximo Paso: Solicitamos aprobación para piloto de tutorías en 2025-1',
        ha='center', va='center', fontsize=13, fontweight='bold', color=COLOR_POSITIVO)

# Pie
ax.text(0.5, 0.01, 'Presentación: Análisis de Matrícula FACES | Enero 2025',
        ha='center', va='bottom', fontsize=9, color='#94a3b8')

plt.tight_layout()
plt.savefig('slide_5_recomendaciones.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\n Slide guardado como 'slide_5_recomendaciones.png'")

---

## Rúbrica de Evaluación

| Criterio | 0-2 puntos | 3-4 puntos | 5 puntos |
|----------|------------|------------|----------|
| **Hook** | Ausente o genérico | Presente pero no impactante | Dato sorprendente que captura atención |
| **Estructura 3C** | Sin estructura clara | Parcialmente estructurado | Contexto-Conflicto-Conclusión claros |
| **Hallazgos** | <2 hallazgos | 2-3 hallazgos básicos | 3+ hallazgos con Afirmación+Evidencia+Implicación |
| **Visualizaciones** | Títulos genéricos, sin anotaciones | Algunas mejoras aplicadas | Todas las técnicas de storytelling aplicadas |
| **Recomendaciones** | Vagas o ausentes | Parcialmente específicas | SMART: Específicas, Medibles, Alcanzables, Relevantes, Temporales |
| **Código** | No funciona | Funciona con errores | Funciona completamente y es reproducible |

**Total: 30 puntos**

---

## Archivos Generados

Este notebook genera los siguientes archivos:

1. `hallazgo_1_tendencia.png` - Gráfico de tendencia con proyección
2. `hallazgo_2_carreras.png` - Comparación de cambio por carrera
3. `hallazgo_3_semestres.png` - Patrón por semestre
4. `slide_5_recomendaciones.png` - Slide de cierre con recomendaciones